Import Depdendencies

In [59]:
import openai
import pandas as pd
import cohere
from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

create quadrant collection for hybrid search

In [60]:
qdrant_client = QdrantClient(url="http://localhost:6333")

Embedding Function

In [61]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [62]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

Preprocessing functions

Hybrid Search with weighted RRF

In [63]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=[3,1])),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [64]:
results = retrieve_data("Can I get some speakers?", k=20)

In [65]:
results

{'retrieved_context_ids': ['B0C1YC2D9Q',
  'B0CDL4Q7WL',
  'B0C996WY16',
  'B09TQ23FKC',
  'B0B6HSDGML',
  'B0BRQ2XGH3',
  'B0C61WKW64',
  'B0CG8ZH1PC',
  'B0C4NJPN4Q',
  'B0CB3CLQYS',
  'B09XBFN4NR',
  'B09TKM2R9K',
  'B0B7QSQ1SJ',
  'B0BRB22GL4',
  'B0BHP9X5MJ',
  'B0BN8H8X2Z',
  'B09YLSP3L3',
  'B0B7PB7NP6',
  'B09SNYHYV7',
  'B09DKGPST6'],
 'retrieved_context': ['Howonder MY-26 Speaker Wall Mounts, Wall Speaker Mount Bracket Compatible with Bookshelf Speaker，PA Speakers,Column Speakers (2 Packs Black) 【Convenient installation】: The speaker mounts can be installed on the wall, which can be tilted up and down by 90°, and the level can be adjusted 180°, so that your speakers can get a perfect angle of sound 【Safe and reliable】: Made of solid steel, it can withstand additional loads of up to 100 pounds. 【Universal and compatible】: This speaker wall mount is almost compatible with most 6" 8" 10" 12" 15" speakers on the market 【TIPS & NOTES】: Please double check that the size and appeara

ReRanking

In [66]:
cohere_client = cohere.ClientV2()

In [67]:
to_rerank = results["retrieved_context"]

In [68]:
to_rerank

['Howonder MY-26 Speaker Wall Mounts, Wall Speaker Mount Bracket Compatible with Bookshelf Speaker，PA Speakers,Column Speakers (2 Packs Black) 【Convenient installation】: The speaker mounts can be installed on the wall, which can be tilted up and down by 90°, and the level can be adjusted 180°, so that your speakers can get a perfect angle of sound 【Safe and reliable】: Made of solid steel, it can withstand additional loads of up to 100 pounds. 【Universal and compatible】: This speaker wall mount is almost compatible with most 6" 8" 10" 12" 15" speakers on the market 【TIPS & NOTES】: Please double check that the size and appearance match your speaker before purchasing. 【Service and Guarantee】:Howonder provides a 12-month quality guarantee, if the product has quality problems, please contact us. Note: The complimentary screws are only suitable for use on solid walls and brick walls. They cannot be used on plaster walls and wood panels.',
 'JAUYXIAN Bluetooth Speaker with Two Wireless Microp

In [70]:
query="Can I get some speakers"

In [71]:
response = cohere_client.rerank(
    model="rerank-v5.0-pro",
    query=query,
    documents=to_rerank,
    top_n=20
)

In [72]:
response

V2RerankResponse(id='43a52464-fcba-47bd-b881-3c626ddfd6fc', results=[V2RerankResponseResultsItem(index=9, relevance_score=0.7182057, relevant_spans=None), V2RerankResponseResultsItem(index=17, relevance_score=0.712455, relevant_spans=None), V2RerankResponseResultsItem(index=4, relevance_score=0.7110523, relevant_spans=None), V2RerankResponseResultsItem(index=5, relevance_score=0.69967955, relevant_spans=None), V2RerankResponseResultsItem(index=18, relevance_score=0.69958013, relevant_spans=None), V2RerankResponseResultsItem(index=2, relevance_score=0.6896875, relevant_spans=None), V2RerankResponseResultsItem(index=19, relevance_score=0.688705, relevant_spans=None), V2RerankResponseResultsItem(index=1, relevance_score=0.67070895, relevant_spans=None), V2RerankResponseResultsItem(index=7, relevance_score=0.658038, relevant_spans=None), V2RerankResponseResultsItem(index=13, relevance_score=0.646352, relevant_spans=None), V2RerankResponseResultsItem(index=11, relevance_score=0.6455124, rel

In [73]:
reranked_results = [to_rerank[result.index] for result in response.results]

In [74]:
reranked_results

["neza Portable Bluetooth Speaker, 20W HD Loud Stereo Sound Wireless Speaker, 18H Playtime Bluetooth Speakers with RGB Flashing Lights, Bluetooth 5.1, IPX7 Waterproof Speakers for Travel/Home/Purple Vibrant Light and Effects 360° Stereo: The portable bluetooth speaker is equipped with color RGB light effects, the gradient lights flash, phase and shine to the beat of the music, creating a vibrant ambiance. The wireless speaker features two built-in 10W speakers, is the best choice for those who need high-quality sound for large areas, such as tailgate parties and neighborhood barbecues. Portable speaker sounds beautiful, with smooth vocals, ample bass, and surprising ambience. IPX7 Waterproof and Portable Design: This waterproof bluetooth speaker is IPX7 waterproof, and rugged speakers suitable for outdoor adventures. These makes waterproof speaker safe in the rain or in the shower. It about 6.69 inches high and 1.01 pounds, it‘s the type of water proof speaker you can just toss in a ba